In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import LabelEncoder, StandardScaler
import optuna, warnings, subprocess, os, gc
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Environment ───────────────────────────────────────────────────────────────
IS_KAGGLE  = os.path.exists("/kaggle/input")
DATA_DIR   = "/kaggle/input/playground-series-s6e3/" if IS_KAGGLE else "../data/"
SUB_DIR    = "/kaggle/working/"                       if IS_KAGGLE else "../"
# Upload nb10/nb09/nb13 submission CSVs as a Kaggle dataset named "churn-stacking-inputs"
CSV_DIR    = "/kaggle/input/churn-stacking-inputs/"   if IS_KAGGLE else "../"

import subprocess
def has_gpu():
    try: return subprocess.run(["nvidia-smi"], capture_output=True, timeout=5).returncode == 0
    except: return False

USE_GPU = IS_KAGGLE or has_gpu()
DEVICE  = "cuda" if USE_GPU else "cpu"

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"GPU         : {'Enabled ✓' if USE_GPU else 'CPU only'}")
print(f"XGBoost     : {xgb.__version__}  |  LightGBM : {lgb.__version__}")

# ── Stacking Settings ─────────────────────────────────────────────────────────
N_SPLITS = 10          # folds for generating OOF from base models
SEED     = 42
TARGET   = "Churn"


In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print(f"Train : {train.shape}  |  Test : {test.shape}")

train[TARGET] = (train[TARGET] == "Yes").astype(int)
churn_rate    = train[TARGET].mean()
scale_pos_w   = round((1 - churn_rate) / churn_rate, 4)
print(f"Churn rate : {churn_rate:.4f}  |  scale_pos_weight : {scale_pos_w}")

sub = test[["id"]].copy()

# Joint preprocessing
combined  = pd.concat([train.drop(TARGET, axis=1), test], axis=0).reset_index(drop=True)
train_idx = range(len(train))
test_idx  = range(len(train), len(train) + len(test))

num_cols = combined.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in combined.select_dtypes(include="object").columns if c != "id"]

for c in num_cols: combined[c].fillna(combined[c].median(), inplace=True)
for c in cat_cols: combined[c].fillna("Missing", inplace=True)

print(f"Numeric cols     : {len(num_cols)}")
print(f"Categorical cols : {len(cat_cols)} → {cat_cols}")


In [ ]:
# Existing FE (proven from nb10 — do NOT add extra noise)
num_base = ["tenure", "MonthlyCharges", "TotalCharges"]
combined["num_sum"]  = combined[num_base].sum(axis=1)
combined["num_mean"] = combined[num_base].mean(axis=1)
combined["num_std"]  = combined[num_base].std(axis=1)
combined["num_max"]  = combined[num_base].max(axis=1)
combined["num_min"]  = combined[num_base].min(axis=1)
combined["Average_Monthly_Actual"] = combined["TotalCharges"] / (combined["tenure"] + 1e-5)
combined["Monthly_diff"]           = combined["MonthlyCharges"] - combined["Average_Monthly_Actual"]
combined["Monthly_ratio"]          = combined["MonthlyCharges"] / (combined["Average_Monthly_Actual"] + 1e-5)

fe_cols = ["num_sum","num_mean","num_std","num_max","num_min",
           "Average_Monthly_Actual","Monthly_diff","Monthly_ratio"]

# Label encode for base models 1 & 2
combined_le = combined.copy()
le = LabelEncoder()
for c in cat_cols:
    combined_le[c] = le.fit_transform(combined_le[c].astype(str))

FEATURES = [c for c in combined_le.columns if c not in ["id", TARGET]]

train_le = combined_le.iloc[train_idx].copy()
test_le  = combined_le.iloc[test_idx].copy()

X      = train_le[FEATURES]
y      = train[TARGET]
X_test = test_le[FEATURES]

print(f"Features : {len(FEATURES)}  |  Train rows : {len(X):,}  |  Test rows : {len(X_test):,}")


In [ ]:
class TargetEncoder:
    """OOF-safe target encoder with smoothing."""
    def __init__(self, cols, alpha=10):
        self.cols, self.alpha = cols, alpha
        self.global_mean, self.maps = None, {}

    def fit(self, X, y):
        self.global_mean = float(y.mean())
        for c in self.cols:
            stats = pd.DataFrame({"y": y.values}, index=X.index)
            stats["cat"] = X[c].values
            agg = stats.groupby("cat")["y"].agg(["sum","count"])
            smoothed = (agg["sum"] + self.alpha * self.global_mean) / (agg["count"] + self.alpha)
            self.maps[c] = smoothed.to_dict()
        return self

    def transform(self, X):
        Xt = X.copy()
        for c in self.cols:
            Xt[c + "_te"] = X[c].map(self.maps[c]).fillna(self.global_mean)
        return Xt

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

# Base model 3 will use original combined (before label encoding) + TE
combined_raw = combined.copy()  # has raw categoricals + FE cols
train_raw = combined_raw.iloc[train_idx].copy()
test_raw  = combined_raw.iloc[test_idx].copy()

print("TargetEncoder ready.")
print(f"Will encode {len(cat_cols)} categorical cols: {cat_cols}")


In [ ]:
# ── Base Model 1: XGB scaledpos+lossguide (nb10 architecture) ─────────────────
XGB_PARAMS_1 = {
    "objective":        "binary:logistic",
    "eval_metric":      "auc",
    "tree_method":      "hist",
    "device":           DEVICE,
    "grow_policy":      "lossguide",
    "scale_pos_weight": scale_pos_w,
    "learning_rate":    0.02,
    "max_depth":        6,
    "max_leaves":       50,
    "min_child_weight": 5,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "colsample_bylevel":0.8,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "gamma":            0.05,
    "verbosity":        0,
    "nthread":          -1,
}
XGB_ROUNDS_1 = 700

# ── Base Model 2: LightGBM GBDT (nb09 architecture) ───────────────────────────
LGB_PARAMS_2 = {
    "objective":         "binary",
    "metric":            "auc",
    "boosting_type":     "gbdt",
    "verbose":           -1,
    "n_jobs":            -1,
    "feature_pre_filter": False,
    "learning_rate":     0.018,
    "num_leaves":        84,
    "max_depth":         10,
    "feature_fraction":  0.65,
    "bagging_fraction":  0.8,
    "bagging_freq":      5,
    "min_child_samples": 20,
    "lambda_l1":         0.019,
    "lambda_l2":         0.006,
}
if USE_GPU:
    LGB_PARAMS_2["device_type"]     = "gpu"
    LGB_PARAMS_2["gpu_platform_id"] = 0
    LGB_PARAMS_2["gpu_device_id"]   = 0
LGB_ROUNDS_2 = 800

# ── Base Model 3: XGB + OOF Target Encoding ────────────────────────────────────
XGB_PARAMS_3 = {**XGB_PARAMS_1}   # same XGB config, different features
XGB_ROUNDS_3 = 700

print("Base model params defined:")
print(f"  BM1 XGB scaledpos+lossguide  | rounds={XGB_ROUNDS_1}")
print(f"  BM2 LGB GBDT tuned           | rounds={LGB_ROUNDS_2}")
print(f"  BM3 XGB + OOF target-enc     | rounds={XGB_ROUNDS_3}")


In [ ]:
def run_base_model(model_id, params, n_rounds, early_stop=100):
    """
    10-fold CV for one base model.
    Returns: oof_preds (N_train,), test_preds (N_test,), fold_aucs
    """
    skf       = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof       = np.zeros(len(X))
    test_sum  = np.zeros(len(X_test))
    fold_aucs = []

    pbar = tqdm(enumerate(skf.split(X, y)), total=N_SPLITS,
                desc=f"BM{model_id}", leave=True)

    for fold, (tr_idx, val_idx) in pbar:
        # ── Feature prep per model type ───────────────────────────────────────
        if model_id in (1, 2):
            # Label-encoded features
            X_tr  = X.iloc[tr_idx];  X_val = X.iloc[val_idx]
            Xt    = X_test.copy()

        elif model_id == 3:
            # OOF target encoding — fit on train fold only (no leakage)
            raw_num_cols = [c for c in train_raw.columns
                            if c not in cat_cols + ["id", TARGET]]
            te = TargetEncoder(cols=cat_cols, alpha=10)
            te.fit(train_raw.iloc[tr_idx][cat_cols],
                   y.iloc[tr_idx])
            tr_enc  = te.transform(train_raw.iloc[tr_idx])
            val_enc = te.transform(train_raw.iloc[val_idx])
            test_enc= te.transform(test_raw)

            te_new_cols = [c + "_te" for c in cat_cols]
            feat_cols   = [c for c in raw_num_cols if c != "id"] + te_new_cols
            X_tr  = tr_enc[feat_cols];  X_val = val_enc[feat_cols]
            Xt    = test_enc[feat_cols]

        y_tr  = y.iloc[tr_idx];  y_val = y.iloc[val_idx]

        # ── Train ─────────────────────────────────────────────────────────────
        if model_id in (1, 3):
            dtrain = xgb.DMatrix(X_tr,  label=y_tr)
            dval   = xgb.DMatrix(X_val, label=y_val)
            dtest  = xgb.DMatrix(Xt)
            model  = xgb.train(
                {**params, "seed": SEED + fold},
                dtrain,
                num_boost_round       = n_rounds,
                evals                 = [(dval, "val")],
                early_stopping_rounds = early_stop,
                verbose_eval          = False,
            )
            vp = model.predict(dval,  iteration_range=(0, model.best_iteration))
            tp = model.predict(dtest, iteration_range=(0, model.best_iteration))

        elif model_id == 2:
            ds_tr  = lgb.Dataset(X_tr,  label=y_tr,  free_raw_data=False)
            ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr, free_raw_data=False)
            model  = lgb.train(
                {**params, "seed": SEED + fold},
                ds_tr,
                num_boost_round = n_rounds,
                valid_sets      = [ds_val],
                callbacks       = [lgb.early_stopping(early_stop, verbose=False)],
            )
            vp = model.predict(X_val, num_iteration=model.best_iteration)
            tp = model.predict(Xt,    num_iteration=model.best_iteration)

        fold_auc = roc_auc_score(y_val, vp)
        fold_aucs.append(fold_auc)
        oof[val_idx] = vp
        test_sum    += tp

        pbar.set_postfix({"fold_auc": f"{fold_auc:.5f}", "oof_auc": f"{roc_auc_score(y, oof if oof.sum()>0 else np.ones(len(y))*0.5):.5f}"})

        del model; gc.collect()

    test_preds = test_sum / N_SPLITS
    final_auc  = roc_auc_score(y, oof)
    print(f"  BM{model_id} OOF AUC : {final_auc:.5f}  "
          f"(fold mean={np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f})")
    return oof, test_preds, fold_aucs


print("=" * 55)
print("LAYER 1 — Generating OOF predictions from 3 base models")
print("=" * 55)

oof1, test1, aucs1 = run_base_model(1, XGB_PARAMS_1, XGB_ROUNDS_1)
oof2, test2, aucs2 = run_base_model(2, LGB_PARAMS_2, LGB_ROUNDS_2)
oof3, test3, aucs3 = run_base_model(3, XGB_PARAMS_3, XGB_ROUNDS_3)

print("\nAll base model OOFs generated ✓")


In [ ]:
# Stack OOF predictions into meta-feature matrix
META_TRAIN = np.column_stack([oof1, oof2, oof3])
META_TEST  = np.column_stack([test1, test2, test3])
y_meta     = y.values

print(f"META_TRAIN shape : {META_TRAIN.shape}")
print(f"META_TEST  shape : {META_TEST.shape}")

# Correlation between base model OOFs — lower = more diversity = better stacking
df_oof = pd.DataFrame(META_TRAIN, columns=["XGB-scaledpos", "LGB-tuned", "XGB-targetenc"])
print("\nOOF Pearson Correlation:")
print(df_oof.corr().round(5))

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.heatmap(df_oof.corr(), annot=True, fmt=".5f", cmap="coolwarm",
            vmin=0.95, vmax=1.0, ax=axes[0], linewidths=0.5)
axes[0].set_title("OOF Correlation (lower = more diverse)", fontsize=12, fontweight="bold")

for col, color in zip(df_oof.columns, ["#2196F3","#4CAF50","#FF9800"]):
    axes[1].hist(df_oof[col], bins=60, alpha=0.6, label=col, color=color)
axes[1].set_xlabel("OOF Predicted Probability")
axes[1].set_ylabel("Count")
axes[1].set_title("OOF Prediction Distributions", fontsize=12, fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
print("=" * 55)
print("LAYER 2 — Training meta-models on OOF")
print("=" * 55)

scaler = StandardScaler()
META_TRAIN_S = scaler.fit_transform(META_TRAIN)
META_TEST_S  = scaler.transform(META_TEST)

# ── Meta-model A: Logistic Regression ────────────────────────────────────────
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
lr.fit(META_TRAIN_S, y_meta)
lr_oof  = lr.predict_proba(META_TRAIN_S)[:, 1]
lr_test = lr.predict_proba(META_TEST_S)[:, 1]
lr_auc  = roc_auc_score(y_meta, lr_oof)
print(f"LogisticRegression  OOF AUC : {lr_auc:.5f}  | coefs: {lr.coef_[0].round(4)}")

# ── Meta-model B: XGBoost (shallow, prevent overfit) ─────────────────────────
xgb_meta_params = {
    "objective":    "binary:logistic",
    "eval_metric":  "auc",
    "max_depth":    2,
    "learning_rate":0.05,
    "n_estimators": 200,
    "subsample":    0.8,
    "seed":         SEED,
    "verbosity":    0,
}
dm_meta_tr = xgb.DMatrix(META_TRAIN, label=y_meta)
dm_meta_te = xgb.DMatrix(META_TEST)

# CV on meta-features to pick best rounds
cv_meta = xgb.cv(
    xgb_meta_params, dm_meta_tr,
    num_boost_round=500, nfold=5, stratified=True,
    early_stopping_rounds=30, seed=SEED, verbose_eval=False,
)
best_meta_rounds = int(cv_meta["test-auc-mean"].idxmax())
print(f"XGB meta — best rounds: {best_meta_rounds}  "
      f"CV AUC: {cv_meta['test-auc-mean'].max():.5f}")

xgb_meta = xgb.train(
    xgb_meta_params, dm_meta_tr,
    num_boost_round=best_meta_rounds, verbose_eval=False,
)
xgb_oof  = xgb_meta.predict(dm_meta_tr)
xgb_test_meta = xgb_meta.predict(dm_meta_te)
xgb_meta_auc = roc_auc_score(y_meta, xgb_oof)
print(f"XGB meta            OOF AUC : {xgb_meta_auc:.5f}")

# ── Meta-model C: Weighted average (simple baseline) ─────────────────────────
w = np.array([0.91445, 0.91419, 0.91415])
w = w / w.sum()
wa_oof  = META_TRAIN @ w
wa_test = META_TEST  @ w
wa_auc  = roc_auc_score(y_meta, wa_oof)
print(f"Score-weighted avg  OOF AUC : {wa_auc:.5f}")

print("\n--- Summary ---")
print(f"  LR meta   : {lr_auc:.5f}")
print(f"  XGB meta  : {xgb_meta_auc:.5f}")
print(f"  WA blend  : {wa_auc:.5f}")
best_meta = max([("LR", lr_auc, lr_test),
                 ("XGB", xgb_meta_auc, xgb_test_meta),
                 ("WA", wa_auc, wa_test)], key=lambda x: x[1])
print(f"  → Best meta-model: {best_meta[0]} ({best_meta[1]:.5f})")


In [ ]:
outputs = {
    "submission_stack_lr.csv":  lr_test,
    "submission_stack_xgb.csv": xgb_test_meta,
    "submission_stack_wa.csv":  wa_test,
}

for fname, preds in outputs.items():
    out = sub.copy()
    out[TARGET] = preds
    fpath = os.path.join(SUB_DIR, fname)
    out.to_csv(fpath, index=False)
    print(f"Saved → {fpath}  | range=[{preds.min():.4f}, {preds.max():.4f}]")

print("\n✓ All stacking submissions saved.")


In [ ]:
import subprocess

COMPETITION = "playground-series-s6e3"

SUBMIT_THESE = [
    ("submission_stack_lr.csv",
     "Stacking L1: XGB-scaledpos+LGB-tuned+XGB-TargetEnc | L2: LogReg meta"),
    ("submission_stack_xgb.csv",
     "Stacking L1: XGB-scaledpos+LGB-tuned+XGB-TargetEnc | L2: XGB depth-2 meta"),
    ("submission_stack_wa.csv",
     "Stacking L1: XGB-scaledpos+LGB-tuned+XGB-TargetEnc | L2: score-weighted avg"),
]

for fname, message in SUBMIT_THESE:
    fpath = os.path.join(SUB_DIR, fname)
    if not os.path.exists(fpath):
        print(f"⚠️  Not found: {fpath}")
        continue
    print(f"\nSubmitting: {fname}")
    r = subprocess.run(
        ["kaggle", "competitions", "submit",
         "-c", COMPETITION, "-f", fpath, "-m", message],
        capture_output=True, text=True
    )
    print(r.stdout.strip())
    if r.stderr.strip(): print("STDERR:", r.stderr.strip())
